In [3]:
import math
import random


class Landschap:
    def __init__(self, kosten, start=0, initial_temperature=10, cooling_rate=0.995, num_iterations=2000):
        self.kosten = kosten
        self.start = start
        self.initial_temperature = initial_temperature
        self.cooling_rate = cooling_rate
        self.num_iterations = num_iterations
        self.n = len(kosten)  # aantal plaatsen

    # -------------------------------------------------------
    # 1. Kostfunctie
    # -------------------------------------------------------
    def cost(self, s):
        """
        Geef de kost op plaats s terug.

        Voorbeeld: cost(3) → 3, cost(10) → 1
        """
        # TODO 1:
        # De kosten staan in de lijst self.kosten.
        # Hint: één regel volstaat.
        
        return self.kosten[s]
        
    # -------------------------------------------------------
    # 2. Buur maken
    # -------------------------------------------------------
    def random_neighbor(self, s):
        """
        Geef willekeurig de plaats links (s - 1) of rechts (s + 1) terug.
        Een buur mag nooit buiten de rij liggen.

        Voorbeeld: random_neighbor(5) → 4 of 6
                   random_neighbor(0) → 1
                   random_neighbor(19) → 18
        """
        # TODO 2:
        # Behandel eerst de twee randen (de eerste en de laatste plaats).
        # De laatste plaats is self.n - 1.

        if s == 0:
            return self.kosten[0]

        elif s == self.n-1:
            return len(self.kosten)


        # TODO 3:
        # In alle andere gevallen: één stap naar links of naar rechts, willekeurig.
        else: 
            return random.choice([-1, 1])

    # -------------------------------------------------------
    # 3. Aanvaardingskans
    # -------------------------------------------------------
    def acceptance_probability(self, delta, T):
        """
        Kans om een slechtere plaats toch te aanvaarden: exp(-delta / T).

        Voorbeeld: acceptance_probability(1, 10)  → ongeveer 0.905
                   acceptance_probability(1, 0.5) → ongeveer 0.135
        """
        # TODO 4:
        # Hint: math.exp(...) en vergeet het minteken niet.
        
        return math.exp(-delta/T)
    
    

    # -------------------------------------------------------
    # 4. Greedy (GEGEVEN, ter vergelijking)
    # -------------------------------------------------------
    def greedy(self):
        """
        Gaat altijd naar de laagste buur en stopt als er geen lagere buur is.
        Blijft dus steken in het eerste dal dat het tegenkomt.
        """
        s = self.start
        while True:
            buren = [b for b in (s - 1, s + 1) if 0 <= b < self.n]
            beste_buur = min(buren, key=self.cost)
            if self.cost(beste_buur) >= self.cost(s):
                return s, self.cost(s)
            s = beste_buur

    # -------------------------------------------------------
    # 5. Simulated annealing
    # -------------------------------------------------------
    def simulated_annealing(self):
        # TODO 5
        # Schrijf het simulated annealing algoritme.
        # Gebruik de parameters zoals ze zijn meegegeven in de klasse (helemaal bovenaan deze file).
        #
        # - start op self.start, met T = self.initial_temperature
        # - herhaal self.num_iterations keer:
        #     - maak een buur
        #     - bereken delta = kost van de buur - kost van de huidige plaats
        #     - betere buur: altijd aanvaarden
        #     - slechtere buur: aanvaarden met kans acceptance_probability(delta, T)
        #     - hou de beste plaats ooit bij
        #     - print elke 250 stappen: stap, T, plaats en kost
        #       print(f"Stap {k}: T = {T:.2f}, plaats {s}, kost {self.cost(s)}")
        #     - koel af: T = T * self.cooling_rate
        # - geef een tuple terug: (beste_plaats, beste_kost)
        
        buur = self.start 
        best_buur = buur
        T = self.initial_temperature
        
        for k in range(self.num_iterations):
            buur_new = self.random_neighbor(buur)
            
            if self.cost(buur_new) < self.cost(buur):
                buur = buur_new
            else:
                delta = self.cost(buur_new) - self.cost(buur)
                P = self.acceptance_probability(delta, T)
                
                T = T * self.cooling_rate
                
                if k % 250 == 0:
                    print(f"Stap {k}: T = {T:.2f}, plaats {buur}, kost {self.cost(buur)}")
                
                if random.random() < P:
                    buur = buur_new
            
            if self.cost(buur) < self.cost(best_buur):
                best_buur = buur
                
        return best_buur, self.cost(best_buur)

# -------------------------------------------------------
# Voorbeeldgebruik
# -------------------------------------------------------
if __name__ == "__main__":
    kosten = [9, 7, 5, 3, 4, 6, 8, 7, 5, 3, 1, 2, 4, 6, 8, 9, 7, 6, 8, 9]
    landschap = Landschap(kosten)

    # Vergelijking: greedy blijft steken in het lokale minimum
    plaats, kost = landschap.greedy()
    print(f"Greedy: plaats {plaats}, kost {kost}\n")

    # Simulated annealing
    plaats, kost = landschap.simulated_annealing()
    print(f"\nSimulated annealing: plaats {plaats}, kost {kost}")


# -------------------------------------------------------
# TODO 6: Korte beschrijving (in commentaar)
# -------------------------------------------------------
# Werkt het? Vindt simulated annealing een betere plaats dan greedy? Waarom?
#
# Wat verandert er met cooling_rate = 0.99? Waarom?
#
# Op welk algoritme lijkt het met initial_temperature = 0.1? Waarom?
#


Greedy: plaats 3, kost 3

Stap 250: T = 3.61, plaats 1, kost 7
Stap 500: T = 1.23, plaats 1, kost 7
Stap 750: T = 0.37, plaats 1, kost 7
Stap 1000: T = 0.10, plaats 1, kost 7
Stap 1250: T = 0.03, plaats 1, kost 7
Stap 1500: T = 0.01, plaats 1, kost 7
Stap 1750: T = 0.00, plaats 1, kost 7

Simulated annealing: plaats 9, kost 3
